In [0]:
#| default_exp parallel

## Parallel notebook operations

Shared locks for MCP and notebook helpers. Calls touching the same notebook are serialized, calls touching different notebooks can proceed independently, and notebook execution uses one global semaphore.

In [0]:
#| export
import threading
from contextlib import contextmanager
from pathlib import Path

In [0]:
#| export
_LOCKS_GUARD = threading.Lock()
_NOTEBOOK_LOCKS = {}
_EXECUTION_SEMAPHORE = threading.Semaphore(1)

In [0]:
#| export
def notebook_key(path):
    "Return a stable lock key for a notebook path."
    if path is None: return None
    return str(Path(path).expanduser().resolve(strict=False))

In [0]:
#| export
def _notebook_lock(key):
    with _LOCKS_GUARD:
        lock = _NOTEBOOK_LOCKS.get(key)
        if lock is None:
            lock = threading.RLock()
            _NOTEBOOK_LOCKS[key] = lock
        return lock

In [0]:
#| export
@contextmanager
def notebook_locks(*paths):
    "Acquire per-notebook locks in a stable order."
    keys = sorted({notebook_key(path) for path in paths if notebook_key(path) is not None})
    locks = [_notebook_lock(key) for key in keys]
    for lock in locks: lock.acquire()
    try:
        yield
    finally:
        for lock in reversed(locks): lock.release()

In [0]:
#| export
@contextmanager
def execution_slot():
    "Serialize notebook execution across parallel MCP calls."
    _EXECUTION_SEMAPHORE.acquire()
    try:
        yield
    finally:
        _EXECUTION_SEMAPHORE.release()

In [0]:
import tempfile
from pathlib import Path as _Path

with tempfile.TemporaryDirectory() as td:
    a = _Path(td) / "a.ipynb"
    b = _Path(td) / "sub" / ".." / "b.ipynb"
    assert notebook_key(a).endswith("a.ipynb")
    assert notebook_key(b).endswith("b.ipynb")
    assert notebook_key(None) is None

In [0]:
import tempfile
import threading
import time
from pathlib import Path as _Path

with tempfile.TemporaryDirectory() as td:
    path = _Path(td) / "same.ipynb"
    entered = []
    def enter_same_lock():
        with notebook_locks(path):
            entered.append(True)
    with notebook_locks(path):
        thread = threading.Thread(target=enter_same_lock)
        thread.start()
        time.sleep(0.05)
        assert entered == []
    thread.join(timeout=1)
    assert entered == [True]

In [0]:
import threading
import time

active = 0
max_active = 0
guard = threading.Lock()

def hold_execution_slot():
    global active, max_active
    with execution_slot():
        with guard:
            active += 1
            max_active = max(max_active, active)
        time.sleep(0.05)
        with guard:
            active -= 1

threads = [threading.Thread(target=hold_execution_slot) for _ in range(3)]
for thread in threads: thread.start()
for thread in threads: thread.join()
assert max_active == 1